# Lesson 4: ReAct Reasoning Loop — Multi-Step Analysis

## What You'll Learn

1. **ReAct pattern** — Reason + Act in an interleaved loop (the backbone of modern agents)
2. **Multi-step planning** — Agent decomposes complex questions into sub-queries
3. **Iteration control** — Max steps, cost budgets, and loop termination
4. **Tool selection** — Agent picks the right tool (SQL vs Python vs schema) per step
5. **Structured plans** — Agent outputs an explicit plan before executing

---

## The ReAct Pattern — Why It Matters

### Before ReAct: Single-shot agents
```
User question → LLM → Answer (one shot, often wrong for complex questions)
```

### After ReAct: Iterative reasoning
```
User question → LLM THINKS → ACTs (tool call) → OBSERVES result
                    ↑                                    |
                    └────────────────────────────────────┘
                         (loop until answer is complete)
```

**ReAct** (Yao et al., 2022) interleaves reasoning traces with actions. The key insight:
LLMs that *think out loud* about what to do next make fewer mistakes than those that
jump straight to an answer.

### How PydanticAI implements ReAct

PydanticAI's agent loop IS a ReAct loop. When you give an agent tools:

1. **LLM receives**: system prompt + user message + tool schemas
2. **LLM decides**: call a tool OR return final output
3. **If tool call**: PydanticAI executes it, feeds result back to LLM
4. **LLM decides again**: call another tool OR return final output
5. **Repeat** until the LLM returns structured output (not a tool call)

The agent naturally does multi-step reasoning when the system prompt encourages it.
But for *complex* analysis, we can make this explicit with a **planning step**.

---

## Setup

In [ ]:
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import pandas as pd
import duckdb
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, ModelRetry

from analyst.tools.code_executor import execute_code_subprocess
from analyst.tools.schema_inspector import inspect_schema

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
employees_df = pd.read_csv(f"{DATA_DIR}/sample_employees.csv")

print(f"Loaded: sales ({sales_df.shape}), employees ({employees_df.shape})")

---

## Part 1: Implicit ReAct — PydanticAI's Built-in Loop

Let's first see how PydanticAI naturally does multi-step reasoning
when given multiple tools and a complex question.

In [ ]:
@dataclass
class AnalystDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    data_dir: str = ""
    step_log: list[dict] = field(default_factory=list)


class AnalysisResult(BaseModel):
    answer: str = Field(description="Complete answer with specific numbers")
    method: str = Field(description="Tools and approach used")
    key_findings: list[str] = Field(description="Bullet-point findings")
    steps_taken: int = Field(description="Number of tool calls made")
    confidence: float = Field(ge=0.0, le=1.0)


react_agent = Agent(
    "openai:local-model",
    deps_type=AnalystDeps,
    output_type=AnalysisResult,
    system_prompt=(
        "You are a senior data analyst. Think step-by-step.\n\n"
        "STRATEGY (ReAct):\n"
        "1. THINK: What do I need to find out first?\n"
        "2. ACT: Use the most appropriate tool\n"
        "3. OBSERVE: Read the result carefully\n"
        "4. THINK: Do I have enough to answer? If not, what next?\n"
        "5. Repeat until confident, then give your final structured answer.\n\n"
        "For complex questions, break them into smaller sub-queries.\n"
        "Always inspect schema before querying unfamiliar data."
    ),
    retries=3,
)


@react_agent.system_prompt
def table_context(ctx: RunContext[AnalystDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


@react_agent.tool
def inspect_table(ctx: RunContext[AnalystDeps], table_name: str) -> str:
    """Get detailed schema: columns, types, stats, sample values."""
    if table_name not in ctx.deps.tables:
        return f"Not found. Available: {list(ctx.deps.tables.keys())}"
    ctx.deps.step_log.append({"tool": "inspect_table", "table": table_name})
    return inspect_schema(ctx.deps.tables[table_name], table_name)


@react_agent.tool
def query_sql(ctx: RunContext[AnalystDeps], sql: str) -> str:
    """Run SQL query against available tables using DuckDB."""
    ctx.deps.step_log.append({"tool": "query_sql", "sql": sql})
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(sql).fetchdf()
        if len(result) > 50:
            return f"First 50 of {len(result)} rows:\n{result.head(50).to_string()}"
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}\nQuery: {sql}")
    finally:
        conn.close()


@react_agent.tool
def run_python(ctx: RunContext[AnalystDeps], code: str) -> str:
    """Execute Python code in sandbox. pandas/numpy/matplotlib available.
    Use DATA_DIR for file paths. Always print() results."""
    ctx.deps.step_log.append({"tool": "run_python", "code_len": len(code)})
    result = execute_code_subprocess(
        code=code, data_dir=ctx.deps.data_dir, timeout_seconds=30,
    )
    if not result.success:
        raise ModelRetry(f"Code failed:\n{result.stderr}")
    parts = []
    if result.stdout:
        parts.append(result.stdout[:3000])
    if result.generated_files:
        parts.append(f"Charts: {result.generated_files}")
    return "\n".join(parts) or "Done (no output)."


print("ReAct agent ready with 3 tools: inspect_table, query_sql, run_python")

In [ ]:
# -----------------------------------------------------------------
# Complex question that requires multiple steps
# -----------------------------------------------------------------

deps = AnalystDeps(
    tables={"sales": sales_df, "employees": employees_df},
    data_dir=DATA_DIR,
)

result = react_agent.run_sync(
    "Compare Electronics vs Home categories across all metrics: "
    "total revenue, total profit, average profit margin, "
    "number of unique products, and best-selling product in each. "
    "Which category is the better business?",
    deps=deps,
)

print(f"Answer:\n{result.output.answer}\n")
print(f"Method: {result.output.method}")
print(f"\nKey findings:")
for f in result.output.key_findings:
    print(f"  - {f}")
print(f"\nSteps taken: {result.output.steps_taken}")
print(f"Confidence: {result.output.confidence:.0%}")
print(f"API requests: {result.usage().requests}")
print(f"\nStep log:")
for i, step in enumerate(deps.step_log, 1):
    print(f"  {i}. {step['tool']}: {str(step)[:100]}")

### Observing the ReAct loop

Notice how the agent:
1. Inspected the schema first (THINK: what columns exist?)
2. Ran SQL for each metric (ACT: get the data)
3. Compared results (OBSERVE + THINK: which is better?)
4. Synthesized a final answer (RESPOND)

This is **implicit** ReAct — PydanticAI's tool loop naturally creates it.

---

## Part 2: Explicit Planning — Plan-then-Execute

For even more complex questions, we can add an **explicit planning step**.
The agent first outputs a structured plan, then executes each step.

This is the **Plan-and-Solve** pattern (Wang et al., 2023).

### Why plan explicitly?

| Approach | Pros | Cons |
|----------|------|------|
| Implicit (let LLM loop) | Simple, flexible | Can wander, hard to debug |
| Explicit plan first | Debuggable, predictable, auditable | Extra LLM call, plan might be wrong |

Production agents often use **explicit planning** because:
- You can **log** and **review** the plan before execution
- You can **estimate cost** before committing
- Users can **approve or modify** the plan (human-in-the-loop)

In [ ]:
# -----------------------------------------------------------------
# Step 1: The Planner Agent — produces a structured plan
# -----------------------------------------------------------------

class AnalysisStep(BaseModel):
    step_number: int
    description: str = Field(description="What this step does")
    tool: str = Field(description="Which tool to use: 'sql', 'python', or 'inspect'")
    query_hint: str = Field(description="SQL query or code approach for this step")


class AnalysisPlan(BaseModel):
    question: str = Field(description="Original question, restated for clarity")
    steps: list[AnalysisStep] = Field(description="Ordered analysis steps")
    estimated_complexity: int = Field(ge=1, le=10)
    estimated_tool_calls: int = Field(description="How many tool calls this will need")


planner = Agent(
    "openai:local-model",
    deps_type=AnalystDeps,
    output_type=AnalysisPlan,
    system_prompt=(
        "You are a data analysis planner. Given a question and available data, "
        "create a step-by-step plan for answering it.\n\n"
        "RULES:\n"
        "- Each step should be a single, clear action\n"
        "- Choose the right tool: 'inspect' for schema, 'sql' for queries, 'python' for complex analysis\n"
        "- Start with schema inspection if the data structure is unclear\n"
        "- Keep plans concise: 2-5 steps for most questions\n"
        "- Do NOT execute anything — only plan"
    ),
)


@planner.system_prompt
def planner_context(ctx: RunContext[AnalystDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


# Test the planner
plan_result = planner.run_sync(
    "Find the most profitable product-region combination and explain "
    "what makes it stand out compared to others.",
    deps=deps,
)

plan = plan_result.output
print(f"Question: {plan.question}")
print(f"Complexity: {plan.estimated_complexity}/10")
print(f"Estimated tool calls: {plan.estimated_tool_calls}")
print(f"\nPlan:")
for step in plan.steps:
    print(f"  {step.step_number}. [{step.tool}] {step.description}")
    print(f"     Hint: {step.query_hint[:100]}")

In [ ]:
# -----------------------------------------------------------------
# Step 2: The Executor — runs the plan step by step
# -----------------------------------------------------------------

class StepResult(BaseModel):
    step_number: int
    tool_used: str
    output: str
    success: bool


class FinalAnalysis(BaseModel):
    answer: str = Field(description="Complete answer with specific numbers")
    key_findings: list[str]
    confidence: float = Field(ge=0.0, le=1.0)


def execute_plan(plan: AnalysisPlan, deps: AnalystDeps) -> list[StepResult]:
    """Execute each step of the plan and collect results."""
    results = []

    for step in plan.steps:
        print(f"  Executing step {step.step_number}: {step.description[:60]}...")

        try:
            if step.tool == "inspect":
                # Find which table to inspect from the hint
                table = next(
                    (t for t in deps.tables if t in step.query_hint.lower()),
                    list(deps.tables.keys())[0],
                )
                output = inspect_schema(deps.tables[table], table)

            elif step.tool == "sql":
                conn = duckdb.connect()
                for name, df in deps.tables.items():
                    conn.register(name, df)
                # Use the hint as the actual query
                output = conn.execute(step.query_hint).fetchdf().to_string()
                conn.close()

            elif step.tool == "python":
                exec_result = execute_code_subprocess(
                    code=step.query_hint,
                    data_dir=deps.data_dir,
                    timeout_seconds=30,
                )
                output = exec_result.stdout if exec_result.success else f"Error: {exec_result.stderr}"
            else:
                output = f"Unknown tool: {step.tool}"

            results.append(StepResult(
                step_number=step.step_number,
                tool_used=step.tool,
                output=output[:2000],  # Truncate
                success=True,
            ))

        except Exception as e:
            results.append(StepResult(
                step_number=step.step_number,
                tool_used=step.tool,
                output=f"Failed: {e}",
                success=False,
            ))

    return results


# Execute the plan
print("Executing plan...")
step_results = execute_plan(plan, deps)

print(f"\nResults:")
for sr in step_results:
    status = "OK" if sr.success else "FAIL"
    print(f"  Step {sr.step_number} [{status}]: {sr.output[:150]}...")

In [ ]:
# -----------------------------------------------------------------
# Step 3: The Synthesizer — turns step results into final answer
# -----------------------------------------------------------------

synthesizer = Agent(
    "openai:local-model",
    output_type=FinalAnalysis,
    system_prompt=(
        "You are a data analyst. Given the results of a multi-step analysis, "
        "synthesize a clear, complete answer. Include specific numbers. "
        "Be concise but thorough."
    ),
)

# Build context from step results
context = f"Original question: {plan.question}\n\nStep results:\n"
for sr in step_results:
    context += f"\nStep {sr.step_number} ({sr.tool_used}):\n{sr.output[:1000]}\n"

final = synthesizer.run_sync(context)

print(f"Final Answer:\n{final.output.answer}\n")
print("Key findings:")
for f in final.output.key_findings:
    print(f"  - {f}")
print(f"\nConfidence: {final.output.confidence:.0%}")

### The Plan-Execute-Synthesize Pipeline

```
User Question
     |
     v
[PLANNER] → AnalysisPlan (structured steps)
     |
     v
[EXECUTOR] → StepResult[] (tool outputs)
     |
     v
[SYNTHESIZER] → FinalAnalysis (coherent answer)
```

Each stage is a separate agent with a focused job. This is a **pipeline architecture** —
simpler to debug, test, and improve than a single monolithic agent.

---

## Part 3: Production Controls — Guardrails on the Loop

An unconstrained agent loop is dangerous:
- It could run forever (infinite tool calls)
- It could blow your budget (hundreds of LLM calls)
- It could waste time on unproductive approaches

Production agents need **guardrails**.

In [ ]:
# -----------------------------------------------------------------
# Guardrailed agent with iteration limits and cost tracking
# -----------------------------------------------------------------

MODEL_PRICING = {
    "gpt-4o-mini": (0.15, 0.60),  # per 1M tokens: (input, output)
    "gpt-4o": (2.50, 10.00),
    "claude-3-5-haiku-latest": (0.80, 4.00),
}


@dataclass
class GuardedDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    data_dir: str = ""
    # Guardrails
    max_tool_calls: int = 8
    max_cost_usd: float = 0.05
    # Tracking
    tool_call_count: int = 0
    total_tokens: int = 0
    step_log: list[dict] = field(default_factory=list)

    def check_budget(self) -> None:
        """Raise if we've exceeded any limit."""
        if self.tool_call_count >= self.max_tool_calls:
            raise RuntimeError(
                f"Tool call limit reached ({self.max_tool_calls}). "
                f"Stopping to prevent runaway execution."
            )

    def log_step(self, tool: str, detail: str) -> None:
        self.tool_call_count += 1
        self.step_log.append({
            "step": self.tool_call_count,
            "tool": tool,
            "detail": detail[:200],
        })
        self.check_budget()


guarded_agent = Agent(
    "openai:local-model",
    deps_type=GuardedDeps,
    output_type=AnalysisResult,
    system_prompt=(
        "You are an efficient data analyst. Answer questions using the minimum "
        "number of tool calls needed.\n\n"
        "EFFICIENCY RULES:\n"
        "- Combine multiple aggregations into a single SQL query when possible\n"
        "- Don't inspect schema if the column names are clear from context\n"
        "- Stop as soon as you have enough data to answer confidently\n"
        "- You have a LIMITED budget of tool calls — use them wisely"
    ),
    retries=2,
)


@guarded_agent.system_prompt
def guarded_context(ctx: RunContext[GuardedDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    remaining = ctx.deps.max_tool_calls - ctx.deps.tool_call_count
    lines.append(f"\nBudget: {remaining} tool calls remaining out of {ctx.deps.max_tool_calls}")
    return "\n".join(lines)


@guarded_agent.tool
def guarded_sql(ctx: RunContext[GuardedDeps], sql: str) -> str:
    """Run SQL query. Budget-tracked."""
    ctx.deps.log_step("sql", sql)
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(sql).fetchdf()
        if len(result) > 30:
            return f"First 30 of {len(result)} rows:\n{result.head(30).to_string()}"
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


@guarded_agent.tool
def guarded_inspect(ctx: RunContext[GuardedDeps], table_name: str) -> str:
    """Inspect table schema. Budget-tracked."""
    ctx.deps.log_step("inspect", table_name)
    if table_name not in ctx.deps.tables:
        return f"Not found. Available: {list(ctx.deps.tables.keys())}"
    return inspect_schema(ctx.deps.tables[table_name], table_name)


# Run with guardrails
guarded_deps = GuardedDeps(
    tables={"sales": sales_df, "employees": employees_df},
    data_dir=DATA_DIR,
    max_tool_calls=6,
)

result = guarded_agent.run_sync(
    "Give me a full business overview: total revenue, top product, "
    "best region, average profit margin, and headcount by department.",
    deps=guarded_deps,
)

print(f"Answer:\n{result.output.answer}\n")
print(f"Tool calls used: {guarded_deps.tool_call_count}/{guarded_deps.max_tool_calls}")
print(f"\nExecution trace:")
for step in guarded_deps.step_log:
    print(f"  {step['step']}. [{step['tool']}] {step['detail'][:80]}")

### Key guardrail patterns

| Guardrail | What it prevents | Implementation |
|-----------|-----------------|----------------|
| Max tool calls | Infinite loops | Counter in deps, check before each tool |
| Cost budget | Runaway spending | Token tracking, stop at threshold |
| Timeout per step | Hung tool calls | `timeout_seconds` on code execution |
| Output truncation | Context overflow | Limit tool output to N chars |
| Retry limit | Infinite retry loops | `retries=N` on agent |

---

## Part 4: Comparing Strategies — Which Works Best?

Let's run the same complex question through both approaches and compare.

In [ ]:
COMPLEX_QUESTION = (
    "Identify the top 3 most profitable product-region pairs. "
    "For each, show total revenue, total profit, and profit margin. "
    "Are any of these trending up or down over time?"
)

# --- Approach 1: Implicit ReAct (single agent with tools) ---
deps1 = AnalystDeps(
    tables={"sales": sales_df, "employees": employees_df},
    data_dir=DATA_DIR,
)
start = time.time()
result1 = react_agent.run_sync(COMPLEX_QUESTION, deps=deps1)
time1 = time.time() - start
usage1 = result1.usage()

print("=== IMPLICIT REACT ===")
print(f"Answer: {result1.output.answer[:300]}...")
print(f"Steps: {result1.output.steps_taken} | Tokens: {usage1.input_tokens + usage1.output_tokens}")
print(f"Time: {time1:.1f}s | Confidence: {result1.output.confidence:.0%}")

# --- Approach 2: Plan-Execute-Synthesize ---
start = time.time()
plan_r = planner.run_sync(COMPLEX_QUESTION, deps=deps1)
step_results = execute_plan(plan_r.output, deps1)

context = f"Question: {plan_r.output.question}\n\nResults:\n"
for sr in step_results:
    context += f"Step {sr.step_number} ({sr.tool_used}): {sr.output[:800]}\n\n"
final = synthesizer.run_sync(context)
time2 = time.time() - start

print(f"\n=== PLAN-EXECUTE-SYNTHESIZE ===")
print(f"Answer: {final.output.answer[:300]}...")
print(f"Plan steps: {len(plan_r.output.steps)} | Succeeded: {sum(1 for s in step_results if s.success)}")
print(f"Time: {time2:.1f}s | Confidence: {final.output.confidence:.0%}")

### When to use which approach

| Approach | Best for | Tradeoff |
|----------|----------|----------|
| **Implicit ReAct** | Simple-medium questions, flexibility | Less predictable, harder to debug |
| **Plan-Execute-Synthesize** | Complex multi-step, auditability | More LLM calls, plan can be wrong |
| **Hybrid** (plan for complex, direct for simple) | Production systems | Need a complexity classifier (Lesson 1) |

---

## Summary

| Concept | What | Why |
|---------|------|-----|
| ReAct loop | Think-Act-Observe cycle | Systematic multi-step reasoning |
| Implicit ReAct | PydanticAI's native tool loop | Simple, flexible |
| Explicit planning | Planner agent outputs structured plan | Debuggable, auditable |
| Plan-Execute-Synthesize | 3-stage pipeline | Separation of concerns |
| Guardrails | Max steps, cost budget, timeouts | Prevent runaway agents |
| Step logging | Track every tool call | Debugging, cost analysis |

### Production patterns
1. **Always set max iterations** — Uncapped loops are production incidents
2. **Log every step** — You need full traces for debugging
3. **Budget-aware agents** — Tell the LLM its remaining budget
4. **Separate planning from execution** — Easier to test each stage
5. **Complexity routing** — Simple questions skip planning overhead

**Next: Lesson 5 — Memory (agent remembers context across turns)**

---
## Exercises

1. **Human-in-the-loop** — Show the plan to the user and let them approve/modify before execution.
2. **Adaptive replanning** — If a step fails, have the planner create a new plan using what succeeded.
3. **Parallel execution** — Execute independent steps concurrently with `asyncio.gather`.
4. **Cost estimator** — Before executing, estimate the cost of the plan based on expected token usage.